# LeetCode 241 — Different Ways to Add Parentheses bằng Python

Link đề: https://leetcode.com/problems/different-ways-to-add-parentheses/ · Độ khó: **Medium** · Chủ đề: Recursion, Memoization, Math, String, Dynamic Programming, Parentheses

Skeleton LeetCode:

```python
class Solution:
    def diffWaysToCompute(self, expression: str) -> List[int]:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập: **sinh mọi cách đặt ngoặc đầy đủ dưới dạng chuỗi rồi tính bằng `eval`**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho biểu thức gồm các số và các toán tử `+`, `-`, `*`; trả về **kết quả của mọi cách đặt ngoặc khác nhau** (một danh sách, có thể chứa **giá trị lặp lại**, thứ tự tùy ý).
2. Nhận ra **cấu trúc đệ quy**: mỗi cách đặt ngoặc tương ứng với việc chọn **một toán tử làm phép tính cuối cùng** (toán tử "gốc"), chia biểu thức thành **vế trái** và **vế phải**, mỗi vế lại đặt ngoặc theo mọi cách, rồi **kết hợp** mọi cặp kết quả.
3. Đi từ **chia để trị đệ quy thuần** → **nhớ lại kết quả (memoization) theo chuỗi con** → **memo theo chỉ số trên danh sách đã tách token** → **quy hoạch động từ dưới lên** → **sinh chuỗi có ngoặc rồi `eval` (brute force theo nghĩa đen)**.
4. Hiểu số cách đặt ngoặc với `k` toán tử là **số Catalan** `C_k`, nên kích thước đầu ra (và thời gian) tăng rất nhanh; bài giới hạn độ dài ≤ 20 để tối đa vài nghìn kết quả.
5. Tránh các lỗi kinh điển: thiếu trường hợp cơ sở (chuỗi chỉ gồm một số), loại trùng bằng `set` (làm mất các kết quả lặp mà đề cần), đảo thứ tự hai vế của phép trừ, và tách từng ký tự khiến số nhiều chữ số bị hỏng.

## 2. Đề bài & ví dụ

Cho chuỗi `expression` gồm các số và các toán tử. Hãy trả về **tất cả các kết quả có thể** khi tính biểu thức theo **mọi cách nhóm các số và toán tử** (mọi cách đặt ngoặc). Có thể trả về theo thứ tự bất kỳ.

Các test được sinh sao cho các giá trị đầu ra vừa trong số nguyên 32 bit và số kết quả khác nhau không vượt quá `10⁴`.

**Ví dụ 1**

- Input: `expression = "2-1-1"`
- Output: `[0,2]`
- Giải thích: `((2-1)-1) = 0` và `(2-(1-1)) = 2`.

**Ví dụ 2**

- Input: `expression = "2*3-4*5"`
- Output: `[-34,-14,-10,-10,10]`
- Giải thích: `(2*(3-(4*5))) = -34`, `((2*3)-(4*5)) = -14`, `((2*(3-4))*5) = -10`, `(2*((3-4)*5)) = -10`, `(((2*3)-4)*5) = 10`. Chú ý `-10` xuất hiện **hai lần** vì có hai cách đặt ngoặc khác nhau cho cùng giá trị.

**Ràng buộc:** `1 <= expression.length <= 20`; `expression` gồm chữ số và các toán tử `+`, `-`, `*`; mọi số nguyên trong biểu thức thuộc `[0, 99]`; không có dấu `+` hoặc `-` đứng trước một số để biểu thị dấu.

In [1]:
from typing import List
import random
import re
from functools import cache

examples = [
    ("2-1-1", [0, 2]),
    ("2*3-4*5", [-34, -14, -10, -10, 10]),
]
for expression, expected in examples:
    print(f"expression={expression!r}, kỳ vọng (sắp) {sorted(expected)}")

expression='2-1-1', kỳ vọng (sắp) [0, 2]
expression='2*3-4*5', kỳ vọng (sắp) [-34, -14, -10, -10, 10]


## 3. Kiến thức nền

### 3.1. Đặt ngoặc quyết định thứ tự tính

Phép trừ không có tính kết hợp: `(2-1)-1 = 0` còn `2-(1-1) = 2`. Vậy đặt ngoặc khác nhau cho kết quả khác nhau. Mỗi cách đặt ngoặc đầy đủ xác định chính xác **thứ tự thực hiện các phép tính**.

In [2]:
print((2 - 1) - 1, 2 - (1 - 1))               # 0 2
print((2 * 3) - (4 * 5), 2 * (3 - (4 * 5)))   # -14 -34

0 2
-14 -34


### 3.2. Chia biểu thức tại một toán tử

Với biểu thức có ít nhất một toán tử, chọn **một** toán tử ở vị trí `i` làm phép tính **cuối cùng**: vế trái là `exp[:i]`, vế phải là `exp[i + 1:]`, toán tử là `exp[i]`. Mỗi vế là một biểu thức nhỏ hơn có thể lại đặt ngoặc theo nhiều cách.

In [3]:
exp = "2*3-4*5"
for i, ch in enumerate(exp):
    if ch in "+-*":
        print(f"tách tại vị trí {i} ({ch!r}): trái = {exp[:i]!r}, phải = {exp[i + 1:]!r}")

tách tại vị trí 1 ('*'): trái = '2', phải = '3-4*5'
tách tại vị trí 3 ('-'): trái = '2*3', phải = '4*5'
tách tại vị trí 5 ('*'): trái = '2*3-4', phải = '5'


### 3.3. `str.isdigit()` để nhận biết "chỉ là một số"

`"23".isdigit()` là `True` (số nhiều chữ số vẫn là toàn chữ số), còn `"2*3".isdigit()` là `False`. Điều kiện này làm **trường hợp cơ sở** của đệ quy: nếu chuỗi chỉ là một số thì chỉ có **một** kết quả, chính số đó.

In [4]:
for s in ["2", "23", "99", "2*3", "2-1-1", ""]:
    print(repr(s), "->", s.isdigit())

'2' -> True
'23' -> True
'99' -> True
'2*3' -> False
'2-1-1' -> False
'' -> False


### 3.4. Kết hợp hai danh sách kết quả: hai vòng `for`

Nếu vế trái cho danh sách `left` và vế phải cho `right`, mọi cặp `(a, b)` với `a ∈ left`, `b ∈ right` cho một kết quả `a ∘ b` (với `∘` là toán tử đã chọn). Số kết quả là `len(left) × len(right)`, **gồm cả giá trị trùng nhau** (mỗi cặp ứng với một cách đặt ngoặc riêng).

In [5]:
left, right = [0, 2], [5]
print([a + b for a in left for b in right])                     # [5, 7]
print([a - b for a in left for b in right])                     # [-5, -3]
print([a * b for a in left for b in [3, 3]])                    # [0, 0, 6, 6]: giá trị trùng vẫn được giữ

[5, 7]
[-5, -3]
[0, 0, 6, 6]


### 3.5. Ghi nhớ kết quả (memoization) với `functools.cache`

Đệ quy thuần có thể tính **cùng một chuỗi con nhiều lần**. `@cache` (hoặc một `dict`) lưu kết quả theo tham số lần gọi; lần sau gặp lại cùng chuỗi thì trả ngay kết quả đã lưu. Điều kiện: tham số phải **băm được** (chuỗi, số, tuple).

In [6]:
calls = 0


@cache
def slow_square(x):
    global calls
    calls += 1
    return x * x


for x in [3, 4, 3, 3, 4]:
    slow_square(x)
print("số lần hàm thực sự chạy:", calls)       # 2 (chỉ x = 3 và x = 4)

số lần hàm thực sự chạy: 2


### 3.6. Số Catalan: có bao nhiêu cách đặt ngoặc?

Với `k` toán tử (`k + 1` số) số cách đặt ngoặc đầy đủ là số Catalan `C_k = (2k)! / ((k + 1)! k!)`: `1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862, ...`. Tăng rất nhanh, nên kích thước đầu ra là phần chủ yếu của độ phức tạp.

In [7]:
from math import comb

def catalan(k):
    return comb(2 * k, k) // (k + 1)


print([catalan(k) for k in range(11)])      # [1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]

[1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay `"2-1-1"`: có hai toán tử `-`.

- **Chọn toán tử thứ nhất làm phép cuối:** trái `"2"`, phải `"1-1"`. `"2"` chỉ có `[2]`. `"1-1"` chỉ có một cách: `[0]`. Kết hợp: `2 - 0 = 2`.
- **Chọn toán tử thứ hai làm phép cuối:** trái `"2-1"` → `[1]`, phải `"1"` → `[1]`. Kết hợp: `1 - 1 = 0`.

Tập kết quả `[2, 0]` (sắp `[0, 2]`), đúng với đề.

Với `"2*3-4*5"` có ba toán tử, ba cách chọn phép cuối (`*` đầu, `-`, `*` sau):

| phép cuối | trái | phải | kết quả |
|---|---|---|---|
| `*` đầu | `2` → `[2]` | `3-4*5` → `[-17, -5]` | `2 · (-17) = -34`, `2 · (-5) = -10` |
| `-` | `2*3` → `[6]` | `4*5` → `[20]` | `6 − 20 = -14` |
| `*` sau | `2*3-4` → `[2, -2]` | `5` → `[5]` | `2 · 5 = 10`, `(-2) · 5 = -10` |

Tổng cộng `2 + 1 + 2 = 5` kết quả, đúng `C_3 = 5`.

In [8]:
print("3-4*5 ->", sorted([3 - (4 * 5), (3 - 4) * 5]))       # [-17, -5]
print("2*3-4 ->", sorted([(2 * 3) - 4, 2 * (3 - 4)]))       # [-2, 2]
print("2*3-4*5 ->", sorted([2 * -17, 2 * -5, 6 - 20, 2 * 5, -2 * 5]))

3-4*5 -> [-17, -5]
2*3-4 -> [-2, 2]
2*3-4*5 -> [-34, -14, -10, -10, 10]


## 5. Cách 1 — Brute Force / baseline: chia để trị đệ quy thuần

**Ý tưởng.** Hàm `dfs(exp)` trả về danh sách mọi kết quả của biểu thức `exp`:

1. Nếu `exp` chỉ gồm chữ số: trả `[int(exp)]`.
2. Ngược lại với mỗi toán tử `exp[i]`: `left = dfs(exp[:i])`, `right = dfs(exp[i + 1:])`; với mọi cặp `(a, b)` thêm `a ∘ b` vào kết quả.

Không nhớ kết quả: một chuỗi con có thể bị tính lại nhiều lần.

In [9]:
def diff_ways_recursive(expression: str) -> list[int]:
    def dfs(exp):
        if exp.isdigit():                               # cơ sở: chỉ là một số
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":                             # chọn toán tử i làm phép cuối
                left = dfs(exp[:i])
                right = dfs(exp[i + 1:])
                for a in left:
                    for b in right:
                        if ch == "+":
                            ans.append(a + b)
                        elif ch == "-":
                            ans.append(a - b)
                        else:
                            ans.append(a * b)
        return ans

    return dfs(expression)


for expression, expected in examples:
    print(f"{expression!r} -> {sorted(diff_ways_recursive(expression))} (kỳ vọng {sorted(expected)})")

'2-1-1' -> [0, 2] (kỳ vọng [0, 2])
'2*3-4*5' -> [-34, -14, -10, -10, 10] (kỳ vọng [-34, -14, -10, -10, 10])


**Độ phức tạp (Cách 1).** Gọi `k` là số toán tử. Số kết quả là `C_k` (Catalan). Thời gian xấp xỉ tổng của `C_k` cho mọi chuỗi con; tính lại nhiều lần nên lớn hơn kích thước đầu ra, cỡ `O(k · C_k)` đến `O(4^k / k^{1/2})`. Với `k ≤ 9` (độ dài ≤ 20) là vài nghìn đến vài chục nghìn phép tính, nhanh. **Space `O(C_k)`** cho kết quả (và `O(k)` đệ quy).

**Nút thắt.** Các chuỗi con như `"3-4*5"` xuất hiện nhiều lần trong các nhánh khác nhau của cây đệ quy và bị tính lại mỗi lần. Ghi nhớ kết quả sẽ tránh việc này.

## 6. Ý tưởng thuật toán / bước đột phá

**Bài toán con chồng lặp.** Mỗi lần gọi `dfs(exp)` chỉ phụ thuộc vào **chuỗi con** `exp`. Chuỗi con là đoạn liên tiếp của biểu thức ban đầu nên chỉ có `O(k²)` đoạn khác nhau (với `k + 1` số). Nhưng đệ quy thuần có thể gọi lại cùng đoạn nhiều lần.

Hai cách tránh lặp:

- **Top-down có nhớ:** dùng `@cache` (hoặc `dict`) cho `dfs` — thêm **một dòng**.
- **Bottom-up (quy hoạch động):** tách biểu thức thành danh sách số `nums` và danh sách toán tử `ops`; `dp[i][j]` là danh sách kết quả của đoạn từ số `i` tới số `j`; điền theo **độ dài đoạn tăng dần**.

Lưu ý quan trọng: kết quả của hàm là **danh sách có thể chứa giá trị lặp**; ta **không** được dedupe (xóa trùng) vì mỗi cách đặt ngoặc là một phần tử, như ví dụ 2 có hai `-10`.

Một điểm cẩn trọng khi dùng `@cache`: danh sách trả về được **chia sẻ** giữa các lần gọi; ta chỉ đọc nó (không sửa) nên an toàn.

In [10]:
nums_demo = list(map(int, re.findall(r"\d+", "2*3-4*5")))
ops_demo = re.findall(r"[-+*]", "2*3-4*5")
print("số   :", nums_demo)            # [2, 3, 4, 5]
print("toán tử:", ops_demo)           # ['*', '-', '*']
print("toán tử thứ k nằm giữa số k và số k + 1:")
for k, op in enumerate(ops_demo):
    print(f"  ops[{k}] = {op!r} nằm giữa nums[{k}] = {nums_demo[k]} và nums[{k + 1}] = {nums_demo[k + 1]}")

số   : [2, 3, 4, 5]
toán tử: ['*', '-', '*']
toán tử thứ k nằm giữa số k và số k + 1:
  ops[0] = '*' nằm giữa nums[0] = 2 và nums[1] = 3
  ops[1] = '-' nằm giữa nums[1] = 3 và nums[2] = 4
  ops[2] = '*' nằm giữa nums[2] = 4 và nums[3] = 5


## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ theo chuỗi con (cách của lời giải tham khảo)

**Giải thích.** Giống Cách 1 nhưng thêm `@cache` lên `dfs`. Mỗi chuỗi con chỉ được tính **một lần**; các lần sau dùng lại danh sách đã lưu.

In [11]:
def diff_ways_memo(expression: str) -> list[int]:
    @cache
    def dfs(exp):
        if exp.isdigit():
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":
                left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                for a in left:
                    for b in right:
                        ans.append(a + b if ch == "+" else a - b if ch == "-" else a * b)
        return ans

    return list(dfs(expression))        # trả về bản sao để không lộ danh sách được cache


for expression, expected in examples:
    print(f"{expression!r} -> {sorted(diff_ways_memo(expression))} (kỳ vọng {sorted(expected)})")

'2-1-1' -> [0, 2] (kỳ vọng [0, 2])
'2*3-4*5' -> [-34, -14, -10, -10, 10] (kỳ vọng [-34, -14, -10, -10, 10])


In [12]:
# Code nộp LeetCode cho Cách 2
from functools import cache
from typing import List


class Solution:
    def diffWaysToCompute(self, expression: str) -> List[int]:
        @cache
        def dfs(exp):
            if exp.isdigit():
                return [int(exp)]
            ans = []
            for i, c in enumerate(exp):
                if c in '-+*':
                    left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                    for a in left:
                        for b in right:
                            if c == '-':
                                ans.append(a - b)
                            elif c == '+':
                                ans.append(a + b)
                            else:
                                ans.append(a * b)
            return ans

        return dfs(expression)

**Độ phức tạp Cách 2:** có `O(k²)` chuỗi con khác nhau; với mỗi chuỗi, ta tạo mọi kết hợp của các cách chia. Tổng thời gian **`O(k · C_k)`** cỡ kích thước đầu ra nhân `k` (thực tế gần `O(C_k)` cho các chuỗi con), **Space `O(k² · C_k)` trong trường hợp xấu** cho bộ nhớ đệm (tổng độ dài các danh sách), thường nhỏ với `k ≤ 9`. Nhanh hơn Cách 1 rõ rệt khi `k` lớn.

### Cách 3 — Memo theo chỉ số trên danh sách số/toán tử đã tách

**Giải thích.** Tách biểu thức một lần thành `nums` và `ops` (bằng biểu thức chính quy, tránh cắt chuỗi lặp lại). Định nghĩa `f(i, j)` là danh sách kết quả của đoạn số `i..j`. Nếu `i == j` trả `[nums[i]]`; ngược lại với mỗi `k` từ `i` đến `j − 1` (toán tử `ops[k]` làm phép cuối) kết hợp `f(i, k)` và `f(k + 1, j)`. Khóa nhớ là cặp số nguyên `(i, j)`, rẻ hơn băm chuỗi.

In [13]:
def diff_ways_index_memo(expression: str) -> list[int]:
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    memo = {}

    def f(i, j):
        if i == j:
            return [nums[i]]
        if (i, j) in memo:
            return memo[(i, j)]
        ans = []
        for k in range(i, j):                       # ops[k] là phép cuối: chia giữa số k và số k + 1
            op = ops[k]
            for a in f(i, k):
                for b in f(k + 1, j):
                    ans.append(a + b if op == "+" else a - b if op == "-" else a * b)
        memo[(i, j)] = ans
        return ans

    return list(f(0, len(nums) - 1))


for expression, expected in examples:
    print(f"{expression!r} -> {sorted(diff_ways_index_memo(expression))} (kỳ vọng {sorted(expected)})")

'2-1-1' -> [0, 2] (kỳ vọng [0, 2])
'2*3-4*5' -> [-34, -14, -10, -10, 10] (kỳ vọng [-34, -14, -10, -10, 10])


**Độ phức tạp Cách 3:** **Time `O(k · C_k)`** như Cách 2 nhưng hằng số nhỏ hơn (không cắt chuỗi), **Space `O(k² · C_k)`** trường hợp xấu cho `memo`.

### Cách 4 — Quy hoạch động từ dưới lên

**Giải thích.** Gọi `m = len(nums)`. `dp[i][j]` là danh sách kết quả của đoạn số `i..j`. Khởi tạo `dp[i][i] = [nums[i]]`. Với độ dài `length = 2..m`, với mỗi `i` (`j = i + length − 1`) và mỗi `k` từ `i` đến `j − 1`: kết hợp `dp[i][k]` với `dp[k+1][j]` bằng toán tử `ops[k]`. Đáp án là `dp[0][m − 1]`. Không đệ quy, thứ tự điền theo độ dài tăng dần đảm bảo các đoạn ngắn hơn đã sẵn sàng.

In [14]:
def diff_ways_dp(expression: str) -> list[int]:
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    m = len(nums)
    dp = [[None] * m for _ in range(m)]
    for i in range(m):
        dp[i][i] = [nums[i]]                         # đoạn một số
    for length in range(2, m + 1):                    # độ dài đoạn tăng dần
        for i in range(m - length + 1):
            j = i + length - 1
            cell = []
            for k in range(i, j):                      # ops[k] là phép cuối
                op = ops[k]
                for a in dp[i][k]:
                    for b in dp[k + 1][j]:
                        cell.append(a + b if op == "+" else a - b if op == "-" else a * b)
            dp[i][j] = cell
    return dp[0][m - 1]


for expression, expected in examples:
    print(f"{expression!r} -> {sorted(diff_ways_dp(expression))} (kỳ vọng {sorted(expected)})")

'2-1-1' -> [0, 2] (kỳ vọng [0, 2])
'2*3-4*5' -> [-34, -14, -10, -10, 10] (kỳ vọng [-34, -14, -10, -10, 10])


**Độ phức tạp Cách 4:** **Time `O(k · C_k)`**, **Space `O(k² · C_k)`** trường hợp xấu (cả bảng `dp`). Có thể giảm bằng cách chỉ giữ các đoạn cần thiết, nhưng không đáng với ràng buộc nhỏ.

### Cách 5 — Sinh mọi chuỗi có ngoặc rồi `eval` (brute force theo nghĩa đen)

**Giải thích.** Sinh **chuỗi biểu thức đầy đủ ngoặc** cho từng cách đặt ngoặc: với mỗi toán tử `k` làm phép cuối, ghép `"(" + trái + op + phải + ")"` cho mọi chuỗi trái/phải. Rồi tính từng chuỗi bằng `eval` (an toàn ở đây vì biểu thức chỉ gồm chữ số, `+ - *` và ngoặc do ta tự sinh). Cách này không "biết" gì về phép tính; chỉ dựa vào Python tính biểu thức, nên là **oracle độc lập tuyệt vời**, nhưng chậm.

In [15]:
def all_parenthesizations(nums, ops, i, j):
    """Mọi chuỗi đầy đủ ngoặc cho đoạn số i..j."""
    if i == j:
        return [str(nums[i])]
    out = []
    for k in range(i, j):
        for left in all_parenthesizations(nums, ops, i, k):
            for right in all_parenthesizations(nums, ops, k + 1, j):
                out.append(f"({left}{ops[k]}{right})")
    return out


def diff_ways_eval(expression: str) -> list[int]:
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    return [eval(s) for s in all_parenthesizations(nums, ops, 0, len(nums) - 1)]


print(all_parenthesizations([2, 1, 1], ["-", "-"], 0, 2))          # ['(2-(1-1))', '((2-1)-1)']
for expression, expected in examples:
    print(f"{expression!r} -> {sorted(diff_ways_eval(expression))} (kỳ vọng {sorted(expected)})")

['(2-(1-1))', '((2-1)-1)']
'2-1-1' -> [0, 2] (kỳ vọng [0, 2])
'2*3-4*5' -> [-34, -14, -10, -10, 10] (kỳ vọng [-34, -14, -10, -10, 10])


**Độ phức tạp Cách 5:** sinh `C_k` chuỗi, mỗi chuỗi dài `O(k)`, `eval` mỗi chuỗi `O(k)`: **Time `O(k · C_k)`** nhưng hằng số lớn (phân tích cú pháp mỗi chuỗi), **Space `O(k · C_k)`** cho các chuỗi. Dùng làm chuẩn đối chiếu, không dùng nộp bài.

> Cách 2 là cách nộp bài ngắn nhất. Cách 3 và 4 tránh cắt chuỗi và đệ quy (DP). Cách 1 và 5 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi cây đệ quy của Cách 1 trên `"2*3-4*5"` (có thụt dòng theo độ sâu), và bảng `dp` của Cách 4 trên `"2-1-1"`.

In [16]:
def trace_dfs(exp, depth=0):
    pad = "  " * depth
    if exp.isdigit():
        print(f"{pad}dfs({exp!r}) -> [{exp}] (cơ sở: một số)")
        return [int(exp)]
    print(f"{pad}dfs({exp!r}):")
    ans = []
    for i, ch in enumerate(exp):
        if ch in "+-*":
            print(f"{pad}  tách tại {ch!r}: trái={exp[:i]!r}, phải={exp[i + 1:]!r}")
            left = trace_dfs(exp[:i], depth + 2)
            right = trace_dfs(exp[i + 1:], depth + 2)
            combined = []
            for a in left:
                for b in right:
                    combined.append(a + b if ch == "+" else a - b if ch == "-" else a * b)
            print(f"{pad}  kết hợp {left} {ch} {right} -> {combined}")
            ans.extend(combined)
    print(f"{pad}=> dfs({exp!r}) = {ans}")
    return ans


result = trace_dfs("3-4*5")
assert sorted(result) == [-17, -5]

dfs('3-4*5'):
  tách tại '-': trái='3', phải='4*5'
    dfs('3') -> [3] (cơ sở: một số)
    dfs('4*5'):
      tách tại '*': trái='4', phải='5'
        dfs('4') -> [4] (cơ sở: một số)
        dfs('5') -> [5] (cơ sở: một số)
      kết hợp [4] * [5] -> [20]
    => dfs('4*5') = [20]
  kết hợp [3] - [20] -> [-17]
  tách tại '*': trái='3-4', phải='5'
    dfs('3-4'):
      tách tại '-': trái='3', phải='4'
        dfs('3') -> [3] (cơ sở: một số)
        dfs('4') -> [4] (cơ sở: một số)
      kết hợp [3] - [4] -> [-1]
    => dfs('3-4') = [-1]
    dfs('5') -> [5] (cơ sở: một số)
  kết hợp [-1] * [5] -> [-5]
=> dfs('3-4*5') = [-17, -5]


In [17]:
def trace_dp(expression):
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    m = len(nums)
    dp = [[None] * m for _ in range(m)]
    print(f"{expression!r}: nums={nums}, ops={ops}")
    for i in range(m):
        dp[i][i] = [nums[i]]
        print(f"  dp[{i}][{i}] = {dp[i][i]}")
    for length in range(2, m + 1):
        for i in range(m - length + 1):
            j = i + length - 1
            cell = []
            for k in range(i, j):
                op = ops[k]
                part = [a + b if op == "+" else a - b if op == "-" else a * b for a in dp[i][k] for b in dp[k + 1][j]]
                print(f"  dp[{i}][{j}] chia tại ops[{k}]={op!r}: dp[{i}][{k}]={dp[i][k]} {op} dp[{k + 1}][{j}]={dp[k + 1][j]} -> {part}")
                cell.extend(part)
            dp[i][j] = cell
            print(f"  => dp[{i}][{j}] = {cell}")
    return dp[0][m - 1]


assert sorted(trace_dp("2-1-1")) == [0, 2]

'2-1-1': nums=[2, 1, 1], ops=['-', '-']
  dp[0][0] = [2]
  dp[1][1] = [1]
  dp[2][2] = [1]
  dp[0][1] chia tại ops[0]='-': dp[0][0]=[2] - dp[1][1]=[1] -> [1]
  => dp[0][1] = [1]
  dp[1][2] chia tại ops[1]='-': dp[1][1]=[1] - dp[2][2]=[1] -> [0]
  => dp[1][2] = [0]
  dp[0][2] chia tại ops[0]='-': dp[0][0]=[2] - dp[1][2]=[0] -> [2]
  dp[0][2] chia tại ops[1]='-': dp[0][1]=[1] - dp[2][2]=[1] -> [0]
  => dp[0][2] = [2, 0]


Bảng tóm tắt (Cách 4, `"2-1-1"`, `nums = [2, 1, 1]`, `ops = ['-', '-']`):

| đoạn | chia tại | kết hợp | kết quả |
|---|---|---|---|
| `dp[0][0]`, `dp[1][1]`, `dp[2][2]` | cơ sở | | `[2]`, `[1]`, `[1]` |
| `dp[0][1]` (`2-1`) | `ops[0] = '-'` | `[2] - [1]` | `[1]` |
| `dp[1][2]` (`1-1`) | `ops[1] = '-'` | `[1] - [1]` | `[0]` |
| `dp[0][2]` (`2-1-1`) | `ops[0]` | `dp[0][0] - dp[1][2] = [2] - [0]` | `2` |
| | `ops[1]` | `dp[0][1] - dp[2][2] = [1] - [1]` | `0` |

Kết quả `dp[0][2] = [2, 0]`, sắp `[0, 2]`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (đệ quy có nhớ theo chuỗi con). Đúng class `Solution`, đúng method `diffWaysToCompute(self, expression: str) -> List[int]`.

In [18]:
from functools import cache
from typing import List


class Solution:
    def diffWaysToCompute(self, expression: str) -> List[int]:
        @cache
        def dfs(exp):
            if exp.isdigit():                          # cơ sở: chỉ là một số
                return [int(exp)]
            ans = []
            for i, c in enumerate(exp):
                if c in "-+*":                         # chọn toán tử i làm phép tính cuối
                    left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                    for a in left:
                        for b in right:
                            if c == "-":
                                ans.append(a - b)
                            elif c == "+":
                                ans.append(a + b)
                            else:
                                ans.append(a * b)
            return ans

        return dfs(expression)


for expression, expected in examples:
    assert sorted(Solution().diffWaysToCompute(expression)) == sorted(expected)
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Mỗi cách đặt ngoặc có đúng một "phép tính cuối":** trong một cách đặt ngoặc đầy đủ, phép tính được thực hiện cuối cùng là toán tử ở "gốc" của cây biểu thức; nó chia biểu thức thành hai vế, mỗi vế được đặt ngoặc độc lập. Ngược lại, chọn một toán tử `i` làm gốc rồi đặt ngoặc mỗi vế theo mọi cách cho ra các cách đặt ngoặc **khác nhau** (hai cách chọn gốc khác nhau, hoặc cách đặt ngoặc vế khác nhau, cho cây biểu thức khác nhau). Vậy tập (cách đặt ngoặc) tương ứng 1–1 với cặp (toán tử gốc, cách đặt ngoặc trái, cách đặt ngoặc phải).
- **Số kết quả:** `C(exp) = Σ_i C(trái_i) · C(phải_i)`, đúng công thức đệ quy của số Catalan, nên danh sách kết quả có đúng một phần tử cho mỗi cách đặt ngoặc (gồm cả giá trị lặp).
- **Cơ sở:** chuỗi chỉ gồm chữ số (một số, có thể nhiều chữ số) chỉ có một cách "đặt ngoặc": chính nó.
- **Thứ tự toán hạng:** `a` lấy từ vế **trái**, `b` từ vế phải, và tính `a ∘ b`. Đảo hai vế sẽ sai với phép trừ.
- **Memo đúng:** kết quả của `dfs(exp)` chỉ phụ thuộc vào `exp` (hàm thuần), nên lưu và dùng lại không đổi kết quả. Ta chỉ đọc danh sách được cache, không sửa.
- **DP (Cách 4)** tính cùng các giá trị theo thứ tự độ dài tăng dần; mỗi `dp[i][j]` dùng các đoạn ngắn hơn đã có.

## 11. Phân tích Time / Space Complexity

Gọi `k` là số toán tử (`k + 1` số) và `C_k` là số Catalan (số cách đặt ngoặc).

- **Cách 1 (đệ quy thuần):** Time vượt `C_k` do tính lại chuỗi con (cỡ `O(k · C_k)` hoặc hơn), Space `O(C_k)`.
- **Cách 2 (đệ quy có nhớ theo chuỗi con):** Time `O(k · C_k)`, Space `O(k² · C_k)` xấu nhất (bộ nhớ đệm).
- **Cách 3 (memo theo chỉ số):** Time `O(k · C_k)` với hằng số nhỏ hơn, Space `O(k² · C_k)` xấu nhất.
- **Cách 4 (DP từ dưới lên):** Time `O(k · C_k)`, Space `O(k² · C_k)` xấu nhất.
- **Cách 5 (sinh chuỗi + `eval`):** Time `O(k · C_k)` hằng số lớn, Space `O(k · C_k)`.
- Vì đầu ra có `C_k` phần tử nên không thể nhanh hơn `Ω(C_k)`. Với độ dài ≤ 20 nghĩa là `k ≤ 9`, `C_9 = 4862`: rất nhỏ.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`; so sánh **đa tập** (danh sách đã sắp, vì giá trị lặp có nghĩa) với `oracle` là Cách 5 viết lại độc lập (sinh chuỗi có ngoặc và `eval`). Biểu thức ngẫu nhiên có số 0..99 (cả nhiều chữ số) và `k` toán tử.

In [19]:
sol = Solution()
impls = {
    "recursive": diff_ways_recursive,
    "memo": diff_ways_memo,
    "index_memo": diff_ways_index_memo,
    "dp": diff_ways_dp,
    "eval": diff_ways_eval,
    "submit": sol.diffWaysToCompute,
}


def oracle(expression):
    """Sinh mọi chuỗi đầy đủ ngoặc bằng duyệt chuỗi (không dùng nums/ops) rồi eval."""
    def gen(s):
        if s.isdigit():
            return [s]
        out = []
        for i, ch in enumerate(s):
            if ch in "+-*":
                for l in gen(s[:i]):
                    for r in gen(s[i + 1:]):
                        out.append("(" + l + ch + r + ")")
        return out
    return sorted(eval(t) for t in gen(expression))


def random_expression(k, hi=99):
    parts = [str(random.randint(0, hi))]
    for _ in range(k):
        parts.append(random.choice("+-*"))
        parts.append(str(random.randint(0, hi)))
    return "".join(parts)


def run_all(expression):
    return {name: sorted(fn(expression)) for name, fn in impls.items()}


for expression, expected in examples:
    assert oracle(expression) == sorted(expected)
    for name, got in run_all(expression).items():
        assert got == sorted(expected), (name, expression, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(expression, label, expected=None):
    want = oracle(expression)
    if expected is not None:
        assert want == sorted(expected), (label, "oracle", want, expected)
    for name, got in run_all(expression).items():
        assert got == want, (label, name, got, want)
    print("PASS", label, "->", want if len(want) <= 8 else f"{len(want)} kết quả")


check_all("0", "một số 0", [0])
check_all("7", "một số một chữ số", [7])
check_all("99", "một số hai chữ số (cơ sở isdigit)", [99])
check_all("1+1", "một toán tử", [2])
check_all("5-3", "một phép trừ", [2])
check_all("2*3*4", "chỉ phép nhân: mọi cách cho cùng giá trị", [24, 24])
check_all("1+2+3", "chỉ phép cộng", [6, 6])
check_all("10+5*2", "số nhiều chữ số", [30, 20])
check_all("2-1-1", "ví dụ 1 của đề", [0, 2])
check_all("2*3-4*5", "ví dụ 2 của đề (giá trị -10 lặp hai lần)", [-34, -14, -10, -10, 10])
check_all("99*99*99", "số lớn", [970299, 970299])
check_all("0*5-0", "có số 0", None)
check_all("1-1-1-1-1", "chuỗi trừ: C_4 = 14 kết quả", None)
assert len(oracle("1-1-1-1-1")) == 14 == catalan(4)
check_all("1+1+1+1+1+1+1+1+1+1", "độ dài 19 (9 toán tử, 4862 kết quả)", None)
assert len(oracle("1+1+1+1+1+1+1+1+1+1")) == catalan(9) == 4862

PASS một số 0 -> [0]
PASS một số một chữ số -> [7]
PASS một số hai chữ số (cơ sở isdigit) -> [99]
PASS một toán tử -> [2]
PASS một phép trừ -> [2]
PASS chỉ phép nhân: mọi cách cho cùng giá trị -> [24, 24]
PASS chỉ phép cộng -> [6, 6]
PASS số nhiều chữ số -> [20, 30]
PASS ví dụ 1 của đề -> [0, 2]
PASS ví dụ 2 của đề (giá trị -10 lặp hai lần) -> [-34, -14, -10, -10, 10]
PASS số lớn -> [970299, 970299]
PASS có số 0 -> [0, 0]
PASS chuỗi trừ: C_4 = 14 kết quả -> 14 kết quả


PASS độ dài 19 (9 toán tử, 4862 kết quả) -> 4862 kết quả


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(241)

# Tính chất 1: số kết quả (kể cả trùng) bằng số Catalan C_k
for _ in range(100):
    k = random.randint(0, 7)
    expression = random_expression(k)
    assert len(sol.diffWaysToCompute(expression)) == catalan(k)
print("PASS tính chất: số kết quả = số Catalan C_k")

# Tính chất 2: giá trị cách đặt ngoặc "từ trái sang phải" và "từ phải sang trái" đều nằm trong kết quả
def left_to_right(expression):
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    value = nums[0]
    for op, x in zip(ops, nums[1:]):
        value = value + x if op == "+" else value - x if op == "-" else value * x
    return value

def right_to_left(expression):
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    value = nums[-1]
    for op, x in zip(reversed(ops), reversed(nums[:-1])):
        value = x + value if op == "+" else x - value if op == "-" else x * value
    return value

for _ in range(200):
    expression = random_expression(random.randint(1, 6))
    results = sol.diffWaysToCompute(expression)
    assert left_to_right(expression) in results and right_to_left(expression) in results
print("PASS tính chất: có mặt cách nhóm từ trái sang phải và từ phải sang trái")

# Phản ví dụ: loại trùng bằng set làm mất các cách đặt ngoặc
def sai_dedupe(expression):
    return sorted(set(sol.diffWaysToCompute(expression)))

print("2*3-4*5 -> sai (set):", sai_dedupe("2*3-4*5"), "| đúng:", sorted(sol.diffWaysToCompute("2*3-4*5")))
assert len(sai_dedupe("2*3-4*5")) == 4 and len(sol.diffWaysToCompute("2*3-4*5")) == 5

PASS tính chất: số kết quả = số Catalan C_k
PASS tính chất: có mặt cách nhóm từ trái sang phải và từ phải sang trái
2*3-4*5 -> sai (set): [-34, -14, -10, 10] | đúng: [-34, -14, -10, -10, 10]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(600):
    k = random.randint(0, 6)
    expression = random_expression(k, hi=random.choice([9, 99]))
    want = oracle(expression)
    for name, got in run_all(expression).items():
        assert got == want, (name, expression, got, want)

for trial in range(10):                       # k = 8, 9 (độ dài <= 20): so các cách nhanh với nhau và với oracle
    k = random.choice([8, 9])
    expression = random_expression(k, hi=9)
    if len(expression) > 20:
        continue
    want = oracle(expression)
    for name, got in run_all(expression).items():
        assert got == want, (name, expression)
print("PASS 600+ biểu thức ngẫu nhiên: mọi cách khớp oracle độc lập (đa tập, gồm giá trị lặp)")

PASS 600+ biểu thức ngẫu nhiên: mọi cách khớp oracle độc lập (đa tập, gồm giá trị lặp)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Thiếu trường hợp cơ sở (chuỗi chỉ là một số)

In [23]:
def sai_thieu_co_so(expression):
    def dfs(exp):
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":
                for a in dfs(exp[:i]):
                    for b in dfs(exp[i + 1:]):
                        ans.append(a + b if ch == "+" else a - b if ch == "-" else a * b)
        return ans                               # với exp = "2" (không toán tử) trả []
    return dfs(expression)


print("2-1-1 -> sai:", sai_thieu_co_so("2-1-1"), "| đúng:", sorted(Solution().diffWaysToCompute("2-1-1")))

2-1-1 -> sai: [] | đúng: [0, 2]


**Vì sao sai:** khi chuỗi chỉ là một số thì không có toán tử nào để chia và vòng lặp không thêm gì, nên `dfs("2")` trả `[]`; mọi tích `for a in [] ...` rỗng và kết quả cuối rỗng. Phải trả `[int(exp)]` khi `exp.isdigit()`.

### Lỗi 2: Loại trùng bằng `set` hoặc bỏ giá trị lặp

In [24]:
def sai_dung_set(expression):
    @cache
    def dfs(exp):
        if exp.isdigit():
            return {int(exp)}                    # dùng set: mất giá trị lặp
        ans = set()
        for i, ch in enumerate(exp):
            if ch in "+-*":
                for a in dfs(exp[:i]):
                    for b in dfs(exp[i + 1:]):
                        ans.add(a + b if ch == "+" else a - b if ch == "-" else a * b)
        return ans
    return sorted(dfs(expression))


print("2*3-4*5 -> sai:", sai_dung_set("2*3-4*5"), "| đúng:", sorted(Solution().diffWaysToCompute("2*3-4*5")))

2*3-4*5 -> sai: [-34, -14, -10, 10] | đúng: [-34, -14, -10, -10, 10]


**Vì sao sai:** đề trả về **mọi cách đặt ngoặc**, mỗi cách một giá trị, **kể cả khi các giá trị bằng nhau** (ví dụ `-10` xuất hiện hai lần). `set` gộp chúng làm số phần tử sai (`4` thay vì `5`). Dùng `list`.

### Lỗi 3: Đảo thứ tự hai vế của phép trừ

In [25]:
def sai_dao_ve(expression):
    @cache
    def dfs(exp):
        if exp.isdigit():
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":
                left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                for a in left:
                    for b in right:
                        ans.append(a + b if ch == "+" else b - a if ch == "-" else a * b)   # sai: b - a
        return ans
    return sorted(dfs(expression))


print("2-1-1 -> sai:", sai_dao_ve("2-1-1"), "| đúng:", sorted(Solution().diffWaysToCompute("2-1-1")))

2-1-1 -> sai: [-2, 2] | đúng: [0, 2]


**Vì sao sai:** `a` đến từ vế **trái**, `b` từ vế **phải**; phép trừ không giao hoán nên phải tính `a - b`. Đảo thành `b - a` cho kết quả khác: với `2-1-1` ta thu `[-2, 2]` thay vì `[0, 2]`.

### Lỗi 4: Tách từng ký tự làm hỏng số nhiều chữ số

In [26]:
def sai_tach_ky_tu(expression):
    @cache
    def dfs(exp):
        if len(exp) == 1:                        # sai: coi mỗi ký tự là một số
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":
                for a in dfs(exp[:i]):
                    for b in dfs(exp[i + 1:]):
                        ans.append(a + b if ch == "+" else a - b if ch == "-" else a * b)
        return ans
    return sorted(dfs(expression))


try:
    print("10+5 ->", sai_tach_ky_tu("10+5"))
except ValueError as e:
    print("10+5 -> lỗi:", e)
print("đúng:", sorted(Solution().diffWaysToCompute("10+5")))

10+5 -> []
đúng: [15]


**Vì sao sai:** số có thể có **hai chữ số** (`10`, `99`). Điều kiện cơ sở phải là "toàn chữ số" (`exp.isdigit()`), không phải "độ dài 1". Với `len(exp) == 1` thì `"10"` không bao giờ là cơ sở; đệ quy không tìm thấy toán tử nào trong `"10"` nên trả `[]`, và kết quả cuối cũng rỗng.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Mọi cách đặt ngoặc / mọi cách nhóm" → **chọn phép tính cuối** rồi đệ quy hai vế (giống Matrix Chain, Burst Balloons, Unique BSTs).
- Số cấu hình tăng theo **Catalan**; nếu chỉ cần *đếm* thì có công thức, nếu cần *liệt kê giá trị* thì phải sinh.
- Gặp bài toán con chồng lặp → thêm **`@cache`** (một dòng) hoặc viết DP theo độ dài đoạn.
- Chú ý tách số **nhiều chữ số**: dùng `isdigit()` hoặc `re.findall(r"\d+", s)`.
- Đừng loại trùng nếu đề tính **số cách** hay trả về từng cách; chỉ loại trùng khi đề yêu cầu "các giá trị khác nhau".
- Khi test: oracle tốt là **sinh chuỗi có ngoặc và `eval`** trên biểu thức nhỏ.

## 15. Biến thể của bài

1. **Đếm số cách đặt ngoặc** (không cần liệt kê): số Catalan.
2. **Chỉ các giá trị khác nhau** (loại trùng).
3. **Giá trị lớn nhất / nhỏ nhất** có thể đạt được.
4. **Thêm toán tử `/`** (cẩn thận chia 0 và chia nguyên).
5. **Liệt kê chính các cách đặt ngoặc** (dạng chuỗi), như `all_parenthesizations`.

In [27]:
def count_ways(expression):
    """Biến thể 1: số cách đặt ngoặc = Catalan(số toán tử)."""
    return catalan(len(re.findall(r"[-+*]", expression)))


def distinct_values(expression):
    """Biến thể 2."""
    return sorted(set(Solution().diffWaysToCompute(expression)))


def max_min_value(expression):
    """Biến thể 3: (nhỏ nhất, lớn nhất)."""
    res = Solution().diffWaysToCompute(expression)
    return min(res), max(res)


def diff_ways_with_div(expression):
    """Biến thể 4: thêm '/' (chia nguyên, bỏ qua các nhánh chia cho 0)."""
    @cache
    def dfs(exp):
        if exp.isdigit():
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*/":
                for a in dfs(exp[:i]):
                    for b in dfs(exp[i + 1:]):
                        if ch == "+":
                            ans.append(a + b)
                        elif ch == "-":
                            ans.append(a - b)
                        elif ch == "*":
                            ans.append(a * b)
                        elif b != 0:
                            ans.append(a // b)
        return ans
    return dfs(expression)


def parenthesizations(expression):
    """Biến thể 5: các chuỗi đầy đủ ngoặc."""
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    return sorted(all_parenthesizations(nums, ops, 0, len(nums) - 1))


assert count_ways("2*3-4*5") == 5 and count_ways("7") == 1
assert distinct_values("2*3-4*5") == [-34, -14, -10, 10]
assert max_min_value("2*3-4*5") == (-34, 10)
assert sorted(diff_ways_with_div("8/4/2")) == [1, 4]
assert parenthesizations("2-1-1") == ["((2-1)-1)", "(2-(1-1))"]
random.seed(15)
for _ in range(100):
    e = random_expression(random.randint(0, 6))
    assert count_ways(e) == len(sol.diffWaysToCompute(e))
    assert len(parenthesizations(e)) == count_ways(e)
print("PASS biến thể: đếm, giá trị khác nhau, max/min, chia, liệt kê ngoặc")

PASS biến thể: đếm, giá trị khác nhau, max/min, chia, liệt kê ngoặc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | chọn phép cuối, kết hợp trái × phải | `O(k·C_k)` hoặc hơn (tính lại) | `O(C_k)` | hiểu ý; oracle nhỏ |
| 2. Đệ quy + `@cache` theo chuỗi con | thêm bộ nhớ đệm | `O(k·C_k)` | `O(k²·C_k)` xấu nhất | **cách nộp bài** |
| 3. Memo theo chỉ số | khóa `(i, j)` trên `nums`/`ops` | `O(k·C_k)` hằng số nhỏ | `O(k²·C_k)` | không muốn cắt chuỗi |
| 4. DP từ dưới lên | `dp[i][j]` theo độ dài đoạn | `O(k·C_k)` | `O(k²·C_k)` | tránh đệ quy |
| 5. Sinh chuỗi + `eval` | liệt kê ngoặc đầy đủ rồi tính | `O(k·C_k)` hằng số lớn | `O(k·C_k)` | oracle độc lập |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"23".isdigit()`, `"2+3".isdigit()`, `catalan(3)`, `sorted(Solution().diffWaysToCompute("5"))`, `sorted(Solution().diffWaysToCompute("2+3"))`.

**Bài 2 (tính tay).** Liệt kê mọi cách đặt ngoặc và kết quả của `"3-2*2"` và `"1+2*3"`. Có bao nhiêu kết quả mỗi biểu thức (kể cả trùng)?

**Bài 3 (điền TODO).** Hoàn thành `ways_todo(expression)` theo Cách 2 (đệ quy có `@cache`).

In [28]:
def ways_todo(expression):
    @cache
    def dfs(exp):
        # TODO 1: nếu exp.isdigit() trả [int(exp)]
        # TODO 2: với mỗi vị trí i có exp[i] là toán tử: left = dfs(exp[:i]); right = dfs(exp[i + 1:])
        # TODO 3: với mọi a trong left, b trong right: thêm a (toán tử) b vào ans
        # TODO 4: trả ans
        pass
    return dfs(expression)

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `ways_dp2(expression)` theo Cách 4 (DP theo độ dài đoạn trên `nums` và `ops`). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_value(expression)` trả về giá trị **lớn nhất** có thể đạt được mà **không** sinh mọi kết quả (gợi ý: DP lưu `(min, max)` cho mỗi đoạn vì phép nhân với số âm). (b) Viết `count_ways_equal(expression, target)` đếm số cách đặt ngoặc cho giá trị đúng `target`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `isdigit` đúng khi toàn chữ số; `catalan(3) = 5`; biểu thức một số có một kết quả. Bài 2: mỗi toán tử làm "phép cuối" một lần.
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `5`, `[5]`, `[5]`. Bài 2: `"3-2*2"`: `3-(2*2) = -1`, `(3-2)*2 = 2`: hai kết quả `[-1, 2]`. `"1+2*3"`: `1+(2*3) = 7`, `(1+2)*3 = 9`: `[7, 9]`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `dp[i][i] = [nums[i]]`, rồi `length` từ `2` đến `m`, `k` từ `i` đến `j − 1`. Bài 5(a): `best[i][j] = (min, max)`; với `+`: `min = minL + minR`, `max = maxL + maxR`; với `-`: `min = minL − maxR`, `max = maxL − minR`; với `*`: lấy min/max của bốn tích. Bài 5(b): `list.count(target)` trên kết quả, hoặc DP đếm theo giá trị.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert "23".isdigit() is True and "2+3".isdigit() is False
assert catalan(3) == 5
assert sorted(Solution().diffWaysToCompute("5")) == [5]
assert sorted(Solution().diffWaysToCompute("2+3")) == [5]

# Bài 2
assert sorted(Solution().diffWaysToCompute("3-2*2")) == [-1, 2]
assert sorted(Solution().diffWaysToCompute("1+2*3")) == [7, 9]

# Bài 3
def ways_todo_ans(expression):
    @cache
    def dfs(exp):
        if exp.isdigit():
            return [int(exp)]
        ans = []
        for i, ch in enumerate(exp):
            if ch in "+-*":
                left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                for a in left:
                    for b in right:
                        ans.append(a + b if ch == "+" else a - b if ch == "-" else a * b)
        return ans
    return dfs(expression)

for expression, expected in examples:
    assert sorted(ways_todo_ans(expression)) == sorted(expected)

# Bài 4
def ways_dp2(expression):
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    m = len(nums)
    dp = [[None] * m for _ in range(m)]
    for i in range(m):
        dp[i][i] = [nums[i]]
    for length in range(2, m + 1):
        for i in range(m - length + 1):
            j = i + length - 1
            dp[i][j] = []
            for k in range(i, j):
                for a in dp[i][k]:
                    for b in dp[k + 1][j]:
                        op = ops[k]
                        dp[i][j].append(a + b if op == "+" else a - b if op == "-" else a * b)
    return dp[0][m - 1]

for expression, expected in examples:
    assert sorted(ways_dp2(expression)) == sorted(expected)

# Bài 5(a), 5(b)
def max_value(expression):
    nums = list(map(int, re.findall(r"\d+", expression)))
    ops = re.findall(r"[-+*]", expression)
    m = len(nums)
    best = [[None] * m for _ in range(m)]
    for i in range(m):
        best[i][i] = (nums[i], nums[i])
    for length in range(2, m + 1):
        for i in range(m - length + 1):
            j = i + length - 1
            lo, hi = None, None
            for k in range(i, j):
                (l1, h1), (l2, h2) = best[i][k], best[k + 1][j]
                if ops[k] == "+":
                    cands = [l1 + l2, h1 + h2]
                elif ops[k] == "-":
                    cands = [l1 - h2, h1 - l2]
                else:
                    cands = [l1 * l2, l1 * h2, h1 * l2, h1 * h2]
                lo = min(cands) if lo is None else min(lo, min(cands))
                hi = max(cands) if hi is None else max(hi, max(cands))
            best[i][j] = (lo, hi)
    return best[0][m - 1][1]

def count_ways_equal(expression, target):
    return Solution().diffWaysToCompute(expression).count(target)

random.seed(19)
for _ in range(200):
    e = random_expression(random.randint(0, 6))
    assert max_value(e) == max(Solution().diffWaysToCompute(e))
assert count_ways_equal("2*3-4*5", -10) == 2
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chọn phép tính cuối, chia trái/phải, kết hợp mọi cặp, có nhớ)

```python
class Solution:
    def diffWaysToCompute(self, expression: str) -> List[int]:
        @cache
        def dfs(exp):
            if exp.isdigit():
                return [int(exp)]
            ans = []
            for i, c in enumerate(exp):
                if c in "-+*":
                    left, right = dfs(exp[:i]), dfs(exp[i + 1:])
                    for a in left:
                        for b in right:
                            ans.append(a - b if c == "-" else a + b if c == "+" else a * b)
            return ans
        return dfs(expression)
```

**Checklist**

- [ ] Cơ sở: `exp.isdigit()` (số **nhiều chữ số** vẫn là một số) trả `[int(exp)]`?
- [ ] Mỗi toán tử làm "phép cuối" một lần; trái `exp[:i]`, phải `exp[i + 1:]`?
- [ ] Kết hợp **mọi cặp** `(a, b)`, `a` từ vế trái, `b` từ vế phải, tính `a ∘ b` đúng thứ tự?
- [ ] **Không** loại trùng (giá trị lặp là các cách đặt ngoặc khác nhau)?
- [ ] Dùng `@cache` / `dict` để tránh tính lại chuỗi con?
- [ ] Đã thử: một số (không có toán tử), một toán tử, số hai chữ số, chỉ phép nhân, ví dụ `2*3-4*5` (giá trị lặp)?
- [ ] Đã đối chiếu với oracle sinh chuỗi có ngoặc + `eval` trên biểu thức ngẫu nhiên (so sánh đa tập đã sắp)?
- [ ] Skeleton đúng `class Solution` / `def diffWaysToCompute(self, expression: str) -> List[int]`?